This notebook checks the trivial cases of all correct or none correct (i.e. k = 0 or N) to see whether they meaningfully differ in linear probe performance compared to other cases. If they do then those values will be excluded as Poulis et al. did, but if they do not then they will be included (as the reason of exclusion they cite, which is that they are trivial cases, would be invalidated).

### Load in the Model (Llama-3.1-8B-Instruct)

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", dtype="float16", device_map="cuda")
print(model.device)
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

cuda:0


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>


### Generating Activations

In [32]:
import pandas as pd

# Load in the Factual Datasets
F3_train, F3_test = pd.read_csv("../dataset/F3_train.csv"), pd.read_csv("../dataset/F3_test.csv")
F4_train, F4_test = pd.read_csv("../dataset/F4_train.csv"), pd.read_csv("../dataset/F4_test.csv")
F5_train, F5_test = pd.read_csv("../dataset/F5_train.csv"), pd.read_csv("../dataset/F5_test.csv")


In [33]:
datasets = {
    "F3_train": F3_train, "F3_test": F3_test,
    "F4_train": F4_train, "F4_test": F4_test,
    "F5_train": F5_train, "F5_test": F5_test,
}

for name, df in datasets.items():
    print(f"{name}: {len(df)}")


F3_train: 1398
F3_test: 600
F4_train: 1394
F4_test: 598
F5_train: 1383
F5_test: 593


In [34]:
import torch
from tqdm import tqdm

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def generate_activations(model, statements, layer, with_chat_template=True, batch_size=16):
    statements = list(statements)
    final_token_activations = []

    for i in range(0, len(statements), batch_size):
        statements_temp = statements[i: i+batch_size]
        if with_chat_template:
            messages = [[{"role": "user", "content": s}] for s in statements_temp]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            inputs = tokenizer(statements_temp, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        final_token_activation = outputs.hidden_states[layer][:, -1, :]
        final_token_activations.append(final_token_activation.cpu())

    return torch.cat(final_token_activations, dim=0)

In [35]:
for dataset in tqdm([F3_train, F3_test,
                     F4_test, F4_train,
                     F5_train, F5_test]):
    activations = generate_activations(model, dataset["statement"], 16, with_chat_template=True, batch_size=16)
    dataset["activations_chat"] = list(activations)
    activations = generate_activations(model, dataset["statement"], 16, with_chat_template=False, batch_size=16)
    dataset["activations"] = list(activations)

100%|██████████| 6/6 [01:25<00:00, 14.19s/it]


### Training the Model and Extracting AUROC Scores

In [36]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def train_probe_pytorch_with_logits(activations, labels, device='cuda'):
    (X_train, X_test), (y_train, y_test) = activations, labels

    X_train = torch.stack(list(X_train)).float().numpy()
    X_test = torch.stack(list(X_test)).float().numpy()
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()


    # Mean-center using only the training mean
    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    # Convert to tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]

    # THIS is the entire "model": one linear layer, no bias.
    # w(x) = w^T x, no offset term -> passes through the origin.
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)

    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()  # sigmoid + binary cross-entropy, combined for numerical stability

    for step in range(1000):
        optimizer.zero_grad()
        logits = probe(X_train_t).squeeze(-1)   # w^T x for every example
        loss = loss_fn(logits, y_train_t)
        loss.backward()
        optimizer.step()

    # Evaluate
    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()
    auroc = roc_auc_score(y_test, test_logits)

    return probe, train_mean, auroc, test_logits

In [37]:
probe, train_mean, auroc, test_logits = train_probe_pytorch_with_logits(
    (F3_train["activations"], F3_test["activations"]),
    (F3_train["label"], F3_test["label"])
)

F3_test["pred_logit"] = test_logits

for k, group in F3_test.groupby("stated_k"):
    k_auroc = roc_auc_score(group["label"], group["pred_logit"])
    print(f"stated_k={k}: n={len(group)}, AUROC={k_auroc:.4f}")


stated_k=0: n=192, AUROC=0.9097
stated_k=1: n=195, AUROC=0.7780
stated_k=2: n=213, AUROC=0.9950


In [38]:
probe, train_mean, auroc, test_logits = train_probe_pytorch_with_logits(
    (F4_train["activations"], F4_test["activations"]),
    (F4_train["label"], F4_test["label"])
)

F4_test["pred_logit"] = test_logits

for k, group in F4_test.groupby("stated_k"):
    k_auroc = roc_auc_score(group["label"], group["pred_logit"])
    print(f"stated_k={k}: n={len(group)}, AUROC={k_auroc:.4f}")

stated_k=0: n=96, AUROC=0.2971
stated_k=1: n=95, AUROC=0.5787
stated_k=2: n=95, AUROC=0.6695
stated_k=3: n=98, AUROC=0.8563
stated_k=4: n=106, AUROC=0.8842
stated_k=5: n=108, AUROC=0.9969


In [39]:
k0 = F4_test[F4_test["stated_k"] == 0]

print("Mean pred_logit by label, stated_k=0:")
print(k0.groupby("label")["pred_logit"].agg(["mean", "std", "count"]))

print("\nOverall pred_logit stats, stated_k=0:")
print(k0["pred_logit"].describe())

print("\nSample rows (sorted by pred_logit):")
print(k0[["statement", "label", "actual_k", "pred_logit"]].sort_values("pred_logit").head(10))
print()
print(k0[["statement", "label", "actual_k", "pred_logit"]].sort_values("pred_logit").tail(10))

Mean pred_logit by label, stated_k=0:
           mean       std  count
label                           
0      0.003706  0.148556     43
1     -0.094300  0.058520     53

Overall pred_logit stats, stated_k=0:
count    96.000000
mean     -0.050401
std       0.118455
min      -0.258761
25%      -0.119440
50%      -0.072198
75%      -0.013986
max       0.404012
Name: pred_logit, dtype: float64

Sample rows (sorted by pred_logit):
                                             statement  label  actual_k  \
31   Exactly 0 of the following cities are in Mali:...      1         0   
58   Exactly 0 of the following cities are in Myanm...      1         0   
401  Exactly 0 of the following cities are in Camer...      0         1   
192  Exactly 0 of the following cities are in Canad...      0         1   
48   Exactly 0 of the following cities are in Austr...      0         1   
531  Exactly 0 of the following cities are in Uzbek...      1         0   
342  Exactly 0 of the following cities are i

In [40]:
probe, train_mean, auroc, test_logits = train_probe_pytorch_with_logits(
    (F5_train["activations"], F5_test["activations"]),
    (F5_train["label"], F5_test["label"])
)

F5_test["pred_logit"] = test_logits
F5_test["stated_k1_k2"] = F5_test["stated_k1"].astype(str) + ", " + F5_test["stated_k2"].astype(str)

for k12, group in F5_test.groupby("stated_k1_k2"):
    k_auroc = roc_auc_score(group["label"], group["pred_logit"])
    print(f"stated_k1_k2={k12}: n={len(group)}, AUROC={k_auroc:.4f}")

stated_k1_k2=0, 0: n=23, AUROC=0.0000
stated_k1_k2=0, 1: n=17, AUROC=0.3000
stated_k1_k2=0, 2: n=20, AUROC=0.3100
stated_k1_k2=0, 3: n=26, AUROC=0.7262
stated_k1_k2=0, 4: n=24, AUROC=0.8071
stated_k1_k2=0, 5: n=23, AUROC=0.6894
stated_k1_k2=1, 0: n=20, AUROC=0.5152
stated_k1_k2=1, 1: n=22, AUROC=0.7025
stated_k1_k2=1, 2: n=21, AUROC=0.7692
stated_k1_k2=1, 3: n=23, AUROC=0.7500
stated_k1_k2=1, 4: n=25, AUROC=0.8194
stated_k1_k2=1, 5: n=18, AUROC=1.0000
stated_k1_k2=2, 0: n=25, AUROC=0.5417
stated_k1_k2=2, 1: n=20, AUROC=0.9500
stated_k1_k2=2, 2: n=19, AUROC=0.8333
stated_k1_k2=2, 3: n=19, AUROC=0.5513
stated_k1_k2=2, 4: n=27, AUROC=0.9560
stated_k1_k2=3, 0: n=26, AUROC=0.5893
stated_k1_k2=3, 1: n=19, AUROC=0.9091
stated_k1_k2=3, 2: n=23, AUROC=0.9048
stated_k1_k2=3, 3: n=30, AUROC=0.9815
stated_k1_k2=4, 0: n=26, AUROC=0.6726
stated_k1_k2=4, 1: n=30, AUROC=0.9241
stated_k1_k2=4, 2: n=28, AUROC=0.9479
stated_k1_k2=5, 0: n=19, AUROC=0.5227
stated_k1_k2=5, 1: n=20, AUROC=0.9500


### Per-k breakdown across layers (F4)

Checks whether the stated_k=0 / stated_k=5 inversion is specific to layer 16, or persists across depth. Does a single forward pass per batch and pulls out all requested layers' hidden states at once, rather than re-running the model per layer.

In [41]:
def generate_activations_multilayer(model, statements, layers, with_chat_template=False, batch_size=16):
    """
    Same as generate_activations, but extracts multiple layers from a single
    forward pass instead of re-running the model once per layer.
    Returns: dict {layer: tensor of shape [n_statements, hidden_dim]}
    """
    statements = list(statements)
    per_layer_activations = {layer: [] for layer in layers}

    for i in tqdm(range(0, len(statements), batch_size), leave=False):
        statements_temp = statements[i: i + batch_size]
        if with_chat_template:
            messages = [[{"role": "user", "content": s}] for s in statements_temp]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            inputs = tokenizer(statements_temp, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        for layer in layers:
            final_token_activation = outputs.hidden_states[layer][:, -1, :]
            per_layer_activations[layer].append(final_token_activation.cpu())

    return {layer: torch.cat(chunks, dim=0) for layer, chunks in per_layer_activations.items()}

In [42]:
num_layers = model.config.num_hidden_layers
LAYERS_TO_CHECK = sorted(set([0, 4, 8, 12, 16, 20, 24, 28, num_layers]))
print(f"Model has {num_layers} transformer layers. Checking layers: {LAYERS_TO_CHECK}")

F4_train_multilayer = generate_activations_multilayer(model, F4_train["statement"], LAYERS_TO_CHECK, with_chat_template=False, batch_size=16)
F4_test_multilayer = generate_activations_multilayer(model, F4_test["statement"], LAYERS_TO_CHECK, with_chat_template=False, batch_size=16)

Model has 32 transformer layers. Checking layers: [0, 4, 8, 12, 16, 20, 24, 28, 32]


In [43]:
results = []

for layer in LAYERS_TO_CHECK:
    probe, train_mean, overall_auroc, test_logits = train_probe_pytorch_with_logits(
        (F4_train_multilayer[layer], F4_test_multilayer[layer]),
        (F4_train["label"], F4_test["label"])
    )

    tmp = F4_test[["stated_k", "label"]].copy()
    tmp["pred_logit"] = test_logits

    for k, group in tmp.groupby("stated_k"):
        k_auroc = roc_auc_score(group["label"], group["pred_logit"])
        results.append({"layer": layer, "stated_k": k, "n": len(group), "auroc": k_auroc})

    results.append({"layer": layer, "stated_k": "ALL", "n": len(tmp), "auroc": overall_auroc})

results_df = pd.DataFrame(results)
pivot = results_df.pivot(index="layer", columns="stated_k", values="auroc")
pivot = pivot[[0, 1, 2, 3, 4, 5, "ALL"]]
print(pivot.round(4))

stated_k       0       1       2       3       4       5     ALL
layer                                                           
0         0.5000  0.5000  0.5000  0.5000  0.5000  0.4912  0.4983
4         0.3208  0.4942  0.4750  0.7359  0.7025  0.8342  0.5856
8         0.2523  0.4613  0.5161  0.7797  0.7039  0.9315  0.5983
12        0.1900  0.5320  0.6199  0.8309  0.8482  0.9972  0.7414
16        0.2971  0.5787  0.6695  0.8563  0.8842  0.9969  0.7962
20        0.5494  0.5862  0.6820  0.8746  0.8940  0.9856  0.8148
24        0.8192  0.5791  0.6856  0.8755  0.9034  0.9873  0.8426
28        0.7876  0.5698  0.6923  0.8746  0.9023  0.9842  0.8377
32        0.8574  0.6031  0.7071  0.8705  0.9156  0.9735  0.8487


### Per-k breakdown across layers (F3)

In [44]:
F3_train_multilayer = generate_activations_multilayer(model, F3_train["statement"], LAYERS_TO_CHECK, with_chat_template=False, batch_size=16)
F3_test_multilayer = generate_activations_multilayer(model, F3_test["statement"], LAYERS_TO_CHECK, with_chat_template=False, batch_size=16)

results_f3 = []

for layer in LAYERS_TO_CHECK:
    probe, train_mean, overall_auroc, test_logits = train_probe_pytorch_with_logits(
        (F3_train_multilayer[layer], F3_test_multilayer[layer]),
        (F3_train["label"], F3_test["label"])
    )

    tmp = F3_test[["stated_k", "label"]].copy()
    tmp["pred_logit"] = test_logits

    for k, group in tmp.groupby("stated_k"):
        k_auroc = roc_auc_score(group["label"], group["pred_logit"])
        results_f3.append({"layer": layer, "stated_k": k, "n": len(group), "auroc": k_auroc})

    results_f3.append({"layer": layer, "stated_k": "ALL", "n": len(tmp), "auroc": overall_auroc})

results_f3_df = pd.DataFrame(results_f3)
pivot_f3 = results_f3_df.pivot(index="layer", columns="stated_k", values="auroc")
pivot_f3 = pivot_f3[[0, 1, 2, "ALL"]]
print(pivot_f3.round(4))

stated_k       0       1       2     ALL
layer                                   
0         0.5000  0.5000  0.4951  0.4983
4         0.3562  0.4906  0.6813  0.5039
8         0.2127  0.5792  0.9260  0.5683
12        0.4981  0.6220  0.9813  0.7726
16        0.9097  0.7780  0.9950  0.9108
20        0.9089  0.8583  0.9821  0.9218
24        0.9407  0.9099  0.9787  0.9422
28        0.9352  0.9202  0.9772  0.9434
32        0.9729  0.9779  0.9800  0.9760


### Per-(k1,k2) breakdown across layers (F5)

In [46]:
F5_train_multilayer = generate_activations_multilayer(model, F5_train["statement"], LAYERS_TO_CHECK, with_chat_template=False, batch_size=16)
F5_test_multilayer = generate_activations_multilayer(model, F5_test["statement"], LAYERS_TO_CHECK, with_chat_template=False, batch_size=16)

F5_test_k_pair = F5_test["stated_k1"].astype(str) + ", " + F5_test["stated_k2"].astype(str)

results_f5 = []

for layer in LAYERS_TO_CHECK:
    probe, train_mean, overall_auroc, test_logits = train_probe_pytorch_with_logits(
        (F5_train_multilayer[layer], F5_test_multilayer[layer]),
        (F5_train["label"], F5_test["label"])
    )

    tmp = F5_test[["label"]].copy()
    tmp["stated_k1_k2"] = F5_test_k_pair
    tmp["pred_logit"] = test_logits

    for k12, group in tmp.groupby("stated_k1_k2"):
        k_auroc = roc_auc_score(group["label"], group["pred_logit"])
        results_f5.append({"layer": layer, "stated_k1_k2": k12, "n": len(group), "auroc": k_auroc})

    results_f5.append({"layer": layer, "stated_k1_k2": "ALL", "n": len(tmp), "auroc": overall_auroc})

results_f5_df = pd.DataFrame(results_f5)
pivot_f5 = results_f5_df.pivot(index="layer", columns="stated_k1_k2", values="auroc")

# order columns: all (k1,k2) pairs sorted, then ALL last
k_pair_cols = sorted([c for c in pivot_f5.columns if c != "ALL"])
pivot_f5 = pivot_f5[k_pair_cols + ["ALL"]]
print(pivot_f5.round(4))

stated_k1_k2    0, 0    0, 1  0, 2    0, 3    0, 4    0, 5    1, 0    1, 1  \
layer                                                                        
0             0.5000  0.5000  0.50  0.5000  0.5000  0.5000  0.5000  0.5000   
4             0.4385  0.6429  0.52  0.3810  0.4500  0.4470  0.6061  0.5950   
8             0.1692  0.2714  0.43  0.5714  0.4643  0.8485  0.5859  0.5124   
12            0.0308  0.2000  0.35  0.6607  0.4643  0.7348  0.4343  0.6364   
16            0.0000  0.3000  0.31  0.7262  0.8071  0.6894  0.5152  0.7025   
20            0.0077  0.3429  0.36  0.7024  0.7857  0.6364  0.5354  0.7438   
24            0.0154  0.3429  0.35  0.7560  0.8071  0.6212  0.5253  0.8017   
28            0.1000  0.4429  0.47  0.7679  0.8429  0.6970  0.5354  0.7273   
32            0.5615  0.6000  0.34  0.7560  0.9071  0.6439  0.5152  0.7603   

stated_k1_k2    1, 2    1, 3  ...    3, 0    3, 1    3, 2    3, 3    4, 0  \
layer                         ...                               

### Evaluation of Results

Poulis et al. exclude "trivial" all-or-none cases (k=0 and k=N) from F4 on the grounds that they are uninformative for probing genuine counting behavior. To evaluate this claim on our reconstruction of F3–F5, we trained a linear probe on the final-token residual stream activations at each of nine layers (0, 4, 8, 12, 16, 20, 24, 28, 32) and computed AUROC separately for each stated-count bucket, rather than relying on a single pooled score. This revealed a layer-dependent pattern rather than a fixed property of the boundary cases themselves. At early-to-mid layers, the probe's output tracked the actual number of matching cities almost independently of the stated number — a magnitude-like signal that happens to align with the ground-truth label at the top of the count range (all False examples at k=N necessarily have actual counts below N, so "logit increases with actual count" separates them correctly) but is maximally anti-correlated with the label at the bottom of the range (all True examples at k=0 have the lowest possible actual count, so the same heuristic predicts backwards). This produced AUROC as low as 0.15 for F4's k=0 bucket at layer 12, alongside near-ceiling performance at k=5 in the same layer — evidence of an intermediate representation that has extracted the raw count but not yet compared it against the stated value. By layer 24–28, this inversion fully resolves: F4's k=0 becomes the highest-scoring bucket (AUROC 0.91), on par with or exceeding the interior k values, directly contradicting the premise that boundary cases are trivially easy or otherwise degenerate. F3 shows a milder version of the same transient (a shallow dip to 0.30 at layer 8) before all three k values converge tightly (0.94–0.98) by layer 24 onward, with no meaningful gap between boundary and interior cases at any well-resolved layer. F5's dual-count structure — now including the previously-excluded k1=0/k2=0 boundary — shows a substantially more severe version of the same early-inversion phenomenon than F4, not a milder one. The (0,0) bucket, where neither named country matches any of the six listed cities, is the single most extreme cell across all three datasets: AUROC of 0.00 at layer 16, remaining near-inverted through layer 28, and only reaching 0.56 by layer 32 — never fully resolving within the range of layers tested. Resolution across the k1=0 row is highly uneven: cells paired with a small k2 stay poor throughout ((0,1)=0.30, (0,2)=0.31 at layer 16, improving only marginally by layer 32), while cells paired with a large k2 recover strongly ((0,3)=0.73, (0,4)=0.81, (0,5)=0.69 at layer 16, climbing further at deeper layers). This directly contradicts a naive "verification complexity" account of task difficulty: (0,0) should require only a single mismatched city to falsify and would intuitively be among the easiest cases, yet it is the hardest cell in the entire table, while (0,4) — which intuitively demands more verification effort — is comparatively easy. Nor does the pattern resolve into a clean boundary-versus-interior gradient the way F3 and F4 do: swapping k1 and k2 (e.g., (0,3)=0.73 vs. (3,0)=0.59; (2,3)=0.55 vs. (3,2)=0.90) produces large, direction-inconsistent asymmetries that do not reduce to a single explanatory rule, and which are at least partly attributable to the small per-cell sample size (~20–30 test examples per (k1,k2) pair even after increasing F5's total to 2,000 examples). We therefore treat the qualitative finding — that F5's boundary cases are not trivial, and in fact expose the underlying magnitude-based confound more starkly than F4 does — as well-supported, while treating the finer-grained structure among individual (k1,k2) cells as unresolved given current sample sizes, rather than fitting it to a mechanism the data cannot reliably support.